In [32]:
import os 
repeat_num = 500
mol_name = 'hps'
# index_name= 'hpss'
condition = 'aggregate'
# xyz_name = 'cothagg.xyz'
f= open('run.sh','w')
for idx in range(repeat_num):
    folder_name = f'simulation_{idx:04}'
    nn_dir = '/curie-home/ansatz/data/comp/fssh_chem_ljb_check/AIE_project/_NN-FF_collections'
    nn_tar_xz = f'NN-{mol_name}.tar.xz'
    nn_file = os.path.join(nn_dir, nn_tar_xz)

    _dir = f'/curie-home/ansatz/data/comp/fssh_chem_ljb_check/AIE_project/workspace/{mol_name}/{condition}'
    
    split_folder = os.path.join(_dir, 'split_output')
    # input_xyz = os.path.join(_dir, xyz_name)
    data_json = os.path.join(_dir, 'data.json')

    input_scirpt = os.path.join(_dir, 'input')
    # index_file = os.path.join(_dir, index_name)
    f.write(f'mkdir {folder_name} && cd {folder_name} && cp {nn_file} . && tar -xf {nn_tar_xz} && cp {_dir}/* . && conda run -n ljb pyrai2md input  > run.log && cd ..\n')
        # print(alpha, s)
        # else:
        #         print(f'{s:.2f}, {alpha:.2f}')
f.close()

In [33]:
OMP_NUM = 16
cpu_per_job = OMP_NUM
ncpu = int(cpu_per_job/OMP_NUM)
with open('run.sh','r') as f:
    text_lines = f.readlines()

slice = (0, len(text_lines))
with open('p_run.make', 'w') as f:
    all_str = ''
    for i in range(slice[0], slice[1]):
        all_str = all_str + f'\trun{i}\t'
    f.write(f'all:\t{all_str}\n')
    f.write('\n')
    for i in range(slice[0], slice[1]):
        line = text_lines[i]
        f.write(f'run{i}:\n')
        f.write(f'\t{line}')
        f.write('\n')

    for i in range(int(len(text_lines)/ncpu)):
        batch_str = f'batch_{i}:'
        for j in range(i*ncpu, ncpu*(i+1)):
            batch_str = batch_str + f'\trun{j}'
        f.write(f'{batch_str}\n') 
        f.write(f'\techo batch_{i} calc\n')
    if len(text_lines)%ncpu:
        batch_str = f'batch_remain:'
        for j in range((int(len(text_lines)/ncpu))*ncpu, len(text_lines)):
            batch_str = batch_str + f'\trun{j}'
        f.write(f'{batch_str}\n') 
        f.write(f'\techo batch_remain calc\n')

In [34]:
! rm *.slurm

In [35]:

slurm_str='''#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --qos=normal

lscpu

free -h


source /software/envs/bash.profile
source /software/envs/anaconda3.env 
which python 
conda activate ljb 
which python 

echo "ymzxm"

tmp
mkdir -p $SLURM_JOB_ID
export SCRDIR=$(pwd)/$SLURM_JOB_ID

cd $SCRDIR
cp $SLURM_SUBMIT_DIR/p_run.make .


'''

! rm *.slurm

for i in range(int(len(text_lines)/cpu_per_job*OMP_NUM)):
    batch_str = f'batch_{i}'
    slurm_script = slurm_str + f'make -f p_run.make {batch_str} -j {int(cpu_per_job/OMP_NUM)}\n'
    with open(f'reno_run_{batch_str}.slurm', 'w') as f:
        f.write('#!/bin/bash\n')
        f.write(f'#SBATCH --job-name="ljb_{mol_name}_{condition}_1_{batch_str}" \n')
        f.write(f'#SBATCH --output="ljb_{batch_str}_%j.err" \n')
        f.write(f'#SBATCH --cpus-per-task={cpu_per_job} \n')
        f.write(slurm_script)
        f.write('cp -r * $SLURM_SUBMIT_DIR\n')

if len(text_lines)%cpu_per_job:
    batch_str = f'batch_remain'
    slurm_script = slurm_str + f'make -f p_run.make {batch_str} -j {int(cpu_per_job/OMP_NUM)}\n'
    with open(f'reno_run_{batch_str}.slurm', 'w') as f:
        f.write('#!/bin/bash\n')
        f.write(f'#SBATCH --job-name="ljb_{mol_name}_{condition}_1_{batch_str}" \n')
        f.write(f'#SBATCH --output="ljb_{batch_str}_%j.err" \n')
        f.write(f'#SBATCH --cpus-per-task={cpu_per_job} \n')
        f.write(slurm_script)
        f.write('cp -r * $SLURM_SUBMIT_DIR\n')

rm: cannot remove '*.slurm': No such file or directory
